### Libraries 

In [2]:
!pip install openai

In [3]:
import pandas as pd
import numpy as np
# from tqdm import tqdm
import random
import time
import csv
import json
import openai
import os
from tqdm import tqdm
import time

### Dataset loading and analysis

#### Arguments

In [4]:
arguments = pd.read_csv("data/arguments.csv")
arguments.head()

,Argument ID,Part,Usage,Conclusion,Stance,Premise
0,A01001,usa,train,Entrapment should be legalized,in favor of,if entrapment can serve to more easily capture...
1,A01002,usa,train,We should ban human cloning,in favor of,we should ban human cloning as it will only ca...
2,A01003,usa,train,We should abandon marriage,against,marriage is the ultimate commitment to someone...
3,A01004,usa,train,We should ban naturopathy,against,it provides a useful income for some people
4,A01005,usa,train,We should ban fast food,in favor of,fast food should be banned because it is reall...


In [5]:
count = 0
for conclusion, stance, premise in arguments[["Conclusion", "Stance", "Premise"]].tail(10).itertuples(index=False, name=None):
    if count == 3:
        break
    print("C: {}, S: {}, P: {}".format(conclusion, stance, premise))
    count+=1
    

C: Reservations in India should be based on economic status, S: against, P: In India, it's easy to hide some income resources and to get a false income certificate. Though the situations are changing for the better, it is not that difficult.
C: There should be a ban on Bandhs, S: in favor of, P: Industries are also one of the worst hits. Investors may lose interest to invest in manufacturing sector as bandh by anyone results in the loss to them.
C: There should be a ban on Bandhs, S: in favor of, P: Though right to protest is a fundamental right, people do not have right to take away the fundamental rights of others such as going to their work, doing their personal works etc. Not everyone is against to the changes by government. Some people may support it and may not support the ban. Forcing them to stay at home is undemocratic.


#### Subsetting arguments from USA - case of study

In [6]:
arguments_usa = arguments[arguments["Part"] == "usa"]
print("Dataframe with USA instances has {} arguments".format(arguments_usa.shape[0]))

Dataframe with USA instances has 5020 arguments


#### Values

In [7]:
import json

with open("values.json", "r", encoding="utf-8") as f:
    values_dict = json.load(f)
values_l1_in_json = {item["name"] for item in values_dict["values"]} # value names in json file - level 1

ground_truth = pd.read_csv("data/labels-level1.csv")
df_value_columns = set(ground_truth.columns[1:])  # skip first column (Argument ID)

# all values in json correspond to the 54 value categories from the ground truth


In [8]:
def build_values_prompt_text(values_dict):
    lines = []

    for v in values_dict["values"]:
        name = v["name"]
        vid = v["id"]
        desc = "; ".join(v["descriptions"]) 

        line = f'{name} - {desc}'
        lines.append(line)

    # return '\n'.join(lines)
    return lines

VALUES_LIST = build_values_prompt_text(values_dict)

# len(VALUES_LIST)

## Baseline Predictor of Values (1000 arguments)

In [9]:
arguments_503 = pd.read_csv("data/arguments_503.csv")
# we want to predict the same 503 arguments as BERT and SVM from the reference paper
l_503 = arguments_503['Argument ID'].to_list()
# some more arguments that are not part of the 503 arguments above, to analyse a bigger sample
l_497 = (
    arguments_usa.loc[~arguments_usa["Argument ID"].isin(l_503), "Argument ID"]
    .head(497)
    .tolist()
)
l_1000 = l_503 + l_497
print(f"Hay un total de {len(l_1000)} argumentos a analizar. Sin repeticiones, {len(set(l_1000))}.")
 # comprobar que no hay repeticiones en los argumentos

arguments_1000 = arguments_usa.loc[
    arguments_usa["Argument ID"].isin(l_503) |
    arguments_usa["Argument ID"].isin(l_497)
]

arguments_1000

Hay un total de 1000 argumentos a analizar. Sin repeticiones, 1000.


,Argument ID,Part,Usage,Conclusion,Stance,Premise
0,A01001,usa,train,Entrapment should be legalized,in favor of,if entrapment can serve to more easily capture...
1,A01002,usa,train,We should ban human cloning,in favor of,we should ban human cloning as it will only ca...
2,A01003,usa,train,We should abandon marriage,against,marriage is the ultimate commitment to someone...
3,A01004,usa,train,We should ban naturopathy,against,it provides a useful income for some people
4,A01005,usa,train,We should ban fast food,in favor of,fast food should be banned because it is reall...
...,...,...,...,...,...,...
5015,A25476,usa,test,We should oppose collectivism,in favor of,opposing collectivism is the right thing to do...
5016,A25480,usa,test,We should limit judicial activism,against,activist judges are an important part of bring...
5017,A25486,usa,test,We should subsidize student loans,in favor of,graduates should not have to start their caree...
5018,A25493,usa,test,We should oppose collectivism,against,collectivism is society working for the benefi...


##### Check if the LLM call works

In [10]:
# import base_url and api_key from private files 
from pathlib import Path

file_path = Path('Cognitive-Bias-and-Heuristics-in-Value-Identification\baseline_predictor.ipynb').resolve().parent.parent / "keys.txt"

with open(file_path, "r", encoding="utf-8") as f:
    first_two_lines = [next(f).rstrip("\n") for _ in range(2)]

base_url, api_key = first_two_lines[0].strip(), first_two_lines[1].strip()

# !pip install openai

from openai import OpenAI
def crear_cliente():
    client = OpenAI(
        base_url=base_url,
        api_key=api_key, 
    )
    return client
 
def call_llm(client, model, prompt):
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=model
    )
    return response.choices[0].message.content


In [11]:
client = crear_cliente()
call_llm(client, 'gpt-oss-120b', 'qué es la IA generativa')

'**Qué es la IA generativa**\n\nLa **Inteligencia Artificial generativa** (o *generative AI* en inglés) es una rama de la IA diseñada para crear contenido nuevo y original a partir de los datos con los que ha sido entrenada. En lugar de limitarse a clasificar, reconocer o predecir información (tareas típicas de la IA discriminativa), la IA generativa “produce” salidas que no existían antes: textos, imágenes, música, video, código, diseños, etc.\n\n---\n\n## 1. Principios básicos\n\n| Concepto | Explicación |\n|----------|-------------|\n| **Modelo generativo** | Un algoritmo que aprende la distribución estadística subyacente de un conjunto de datos y, a partir de esa distribución, puede generar muestras nuevas que se asemejan a los datos de entrenamiento. |\n| **Aprendizaje no supervisado (o auto‑supervisado)** | La mayoría de los modelos generativos se entrenan sin etiquetas explícitas; en cambio, aprenden a predecir partes del dato a partir de otras partes (p.ej., predecir la siguien

#### Prompt Engineering and Experimentation -> 1,165 total tokens

In [12]:
# len(['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view'])

In [13]:
### 1,165
BASELINE_VALUE_PREDICTION_PROMPT = """
# ROLE & GOAL
You are an expert annotator. Your task is to determine which of 54 human values could reasonably serve as a justification for a given argument.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK:
Select for the given argument which of 54 justifications one could provide for it. 

# SELECTION GUIDELINES:
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Read the argument and justification. Select 1 (someone could provide the justification for the argument, even if you may disagree) or 0 (the justification makes no sense for the argument).

# WORKFLOW
1. Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. For each value in the VALUES_LIST (in fixed order), If asked 'Why is this good?', might this be their justification? "Because it is good to "value"".
3. Assign 1 or 0 accordingly.


# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key-value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# VALUES_LIST (in fixed order)
 "{VALUES_LIST}"
 
# EXAMPLE OUTPUT (illustrative, not for this input)
{{'Be creative':1, 'Be curious':0, 'Have freedom of thought':0, 'Be choosing own goals':0,..., 'Have the wisdom to accept others':1, 'Be logical':0, 'Have an objective view':0}}

 
# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"

"""

In [14]:
OLD_VALUE_PREDICTION_PROMPT = """
INSTRUCTIONS:
- Select for the given argument which of 54 justifications one could provide for it. 
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Make sure you understand the examples given.
- Read the argument and justification. Select 1 (someone could provide the justification for the argument, even if you may disagree) or 0 (the justification makes no sense for the argument).

EXAMPLES:
Example arguments against "Social media should be banned".

- Argument: We have to be honest. Social media does not make people polite. But it makes our lives easier and more interesting.	
- Select all justifications one could provide: (✔ have a comfortable life) (from "easier lives"), (✔ have pleasure) (also from "easier lives"), (✔  ahaven exciting life) (from "more interesting"), (✔ have a varied life) (also from "more interesting"). But do not select justifications for concessions ((✖ be polite)) or empty phrases ((✖ be honest), (✖ be logical), (✖ have an objective view) for "We have to be honest").
Example arguments in favor of "Social media should be banned".
- Argument: Through social media people can spread biased opinions on topics or misinform the general public.	
- Use the examples for each justification to get a better understanding of the justifications ((✔ have freedom of thought) from reduced misleading influence on people's thoughts). But do not select justifications only because they are connected to the topic in general ((✖ have privacy) for the general threat of social media to privacy: it is not mentioned here).

TASK:
Select for the given argument which of 54 justifications one could provide for it. 

Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
If asked 'Why is this good?', might this be their justification? "Because it is good to "VALUE"".

Complete this task for each of the justifications, that is, each of the 54 human values with their descriptions in the following list.

Justifications (in fixed order): "{VALUES_LIST}"


Return format:
- Return a dictionary of EXACTLY 54 keys and 54 values. Keys correspond to each of the justifications, following the fixed order. The dictionary values must be an integer: 0 or 1.
    - 1 = YES (someone could provide the justification for the argument)
    - 0 = NO (the justification makes no sense for the argument)
- The order MUST match exactly the order of the VALUES list: ['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view']
- Do NOT skip any value and do NOT add text, explanation, or formatting
- the output MUST BE a DICTIONARY 

Example output as dictionary:
{'Be creative':1, 'Be curious':0, 'Have freedom of thought':0, 'Be choosing own goals':0,..., 'Have the wisdom to accept others':1, 'Be logical':0, 'Have an objective view':0}
(with the length of the list being EXACTLY 54 and the order matching the VALUES list)
"""


In [15]:
def run_experiment_biases(client, model, arguments, bias_prompt, output_path):
    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        done_ids = set(existing_df["Argument ID"].tolist())
        results = existing_df.values.tolist()
        print(f"Resuming... {len(done_ids)} arguments already processed (out of {len(arguments)}).")
        if len(done_ids) == len(arguments):
            return True
    else:
        done_ids = set()
        results = []

    segons_llista = []

    for _, row in tqdm(arguments.iterrows(), total=len(arguments)):
        arg_id = row["Argument ID"]

        if arg_id in done_ids:
            continue

        start = time.time()
        
        prompt = bias_prompt.format(
            CONCLUSION=row["Conclusion"],
            STANCE=row["Stance"],
            PREMISE=row["Premise"],
            VALUES_LIST=VALUES_LIST
        )

        vector = call_llm(client, model, prompt)

        end = time.time()
        segons = end - start
        segons_llista.append(segons)

        results.append([arg_id] + [str(vector)] + [segons])

        df = pd.DataFrame(
            results,
            columns=["Argument ID"] + ["values"] + ["seconds"]
        )

        df.to_csv(output_path, index=False)

    return pd.DataFrame(results, columns=["Argument ID"] + ["values"] + ["seconds"])
    

In [16]:
VALUES_LIST

['Be creative - allowing for more creativity or imagination; being more creative; fostering creativity; promoting imagination',
 'Be curious - being the more interesting option; fostering curiosity; making people more keen to learn; promoting discoveries; sparking interest',
 "Have freedom of thought - allowing people to figure things out on their own; allowing people to make up their mind; resulting in less censorship; resulting in less influence on people's thoughts",
 'Be choosing own goals - allowing people to choose what is best for them; allowing people to decide on their life; allowing people to follow their dreams',
 'Be independent - allowing people to plan on their own; resulting in fewer times people have to ask for consent',
 'Have freedom of action - allowing people to be self-determined; allowing people to do things even though this may hurt them in the long run; resulting in more times people can do what they want',
 'Have privacy - allowing for private spaces; allowing 

#### LLM selection

Si el objetivo es seguir instrucciones de texto (instruction following), sin necesidad de programar o analizar imágenes, el orden aproximado sería:
* **Qwen3.6-35B-A3B**	⭐⭐⭐⭐⭐	Uno de los mejores modelos abiertos actuales para seguir instrucciones, razonamiento y conversación. https://huggingface.co/Qwen/Qwen3.6-35B-A3B // https://qwen.ai/blog?id=qwen3.6-35b-a3b

* **Llama 3.3 70B**	⭐⭐⭐⭐⭐	Excelente capacidad conversacional e instruction following. Muy cercano a Qwen, aunque suele ser algo menos preciso en tareas complejas. https://ollama.com/library/llama3.3:70b

* **GPT-OSS-120B**	⭐⭐⭐⭐☆	Muy potente por tamaño (120B), pero en la práctica no siempre supera a Qwen 3.6 en adherencia estricta a instrucciones. https://openai.com/es-ES/index/introducing-gpt-oss/

In [17]:
# llms = ['gpt-oss-120b', 'llama3.3:70b', 'Qwen3.6-35B-A3B']
llms = ['gpt-oss-120b']
output_csv = ['results/baseline_gpt_oss_1000.csv']

#### Run Experiments

In [18]:
client = crear_cliente()
for model, output in zip(llms, output_csv):
    results = run_experiment_biases(client, model, arguments_1000,BASELINE_VALUE_PREDICTION_PROMPT, output)

Resuming... 1000 arguments already processed (out of 1000).


## Cognitive Bias Predictor of Values (503 arguments)

In [19]:
arguments_503.head()

,Argument ID,Part,Usage,Conclusion,Stance,Premise
0,A05039,usa,test,We should subsidize student loans,in favor of,student loans set children up to be valuable c...
1,A05040,usa,test,We should legalize sex selection,against,we should not do this because other countries ...
2,A05041,usa,test,We should oppose collectivism,in favor of,collectivism prevents original thoughts and di...
3,A05042,usa,test,We should ban missionary work,in favor of,missionary work plays on the ignorance of loca...
4,A05044,usa,test,We should limit judicial activism,in favor of,activists judges aren't the ones shaping the l...


### Cognitive Bias Prompt Experts

##### Framing Effect Prompt

In [20]:
FRAMING_EFFECT_PROMPT = """
# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **framing effect bias**.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK
Select for the given argument which of 54 justifications in the VALUE_LIST one could provide for it **considering how framing effects might shape or distort that justification**.
**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is influenced by framing effects.
**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from frame-biased reasoning**? 

# SELECTION GUIDELINES (FRAMING EFFECT-AWARE)
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Read the argument and justification. Select 1 (someone could provide the justification for the argument, even if you may disagree) or 0 (the justification makes no sense for the argument).

# WORKFLOW
1. Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential framing effects** that could influence how this argument is formulated and justified.
   - Is the premise framed in terms of gains (promotion) or losses (prevention)?
   - Does the argument emphasize avoiding negative outcomes (loss aversion) or pursuing positive outcomes?
   - How does the stance ("for"/"against") shape the interpretation of the premise and the values that seem relevant?
3. For each value in the VALUES_LIST (in fixed order), If asked 'Why is this good?', might this be their justification? "Because it is good to "value"" be a coherent justification that **could plausibly arise from frame-biased reasoning**? Consider:
   - Would the frame (gain vs. loss) make this value seem more or less compelling?
   - For a "for" stance, are prevention-focused values (safety, stability, security) being over-selected?
   - For an "against" stance, are promotion-focused values (freedom, creativity, independence) being over-selected?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key-value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# FRAMING EFFECT EXPERT CONTEXT (Mental Model)
- **Framing effect bias** in reasoning is the tendency to make different decisions based on how information is presented (the "frame"), rather than on the objective information itself. People are systematically influenced by whether outcomes are framed as gains (benefits) or losses (costs), and by the stance taken ("for" or "against").
- **Your expertise includes:** Gain vs. loss framing, promotion vs. prevention focus, loss aversion, risk perception differences, valence framing (positive vs. negative), regulatory focus theory, and the tendency to overweight losses relative to equivalent gains.
- **Your mission:** Identify values that a *frame-biased* human would select, not values that a *rational* human would select. The arguer may be influenced by whether the argument is framed as preventing harm (for stance) or promoting freedom (against stance), may overweight losses relative to gains, or may interpret the same premise differently depending on the frame.

# VALUES_LIST (in fixed order)
 "{VALUES_LIST}"
 
# EXAMPLE OUTPUT (illustrative, not for this input)
{{'Be creative':1, 'Be curious':0, 'Have freedom of thought':0, 'Be choosing own goals':0,..., 'Have the wisdom to accept others':1, 'Be logical':0, 'Have an objective view':0}}

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"
"""

##### Motivated Reasoning Prompt

In [21]:
MOTIVATED_REASONING_PROMPT = """
# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **motivated reasoning bias**.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK
Select for the given argument which of 54 justifications in the VALUE_LIST one could provide for it **considering how motivated reasoning might shape or distort that justification**.
**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is influenced by motivated reasoning.
**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from motivated reasoning**? 

# SELECTION GUIDELINES (MOTIVATED REASONING-AWARE)
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Read the argument and justification. Select 1 (someone could provide the justification for the argument, even if you may disagree) or 0 (the justification makes no sense for the argument).

# WORKFLOW
1. Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential motivated reasoning** that could influence how this argument is formulated and justified.
   - What conclusion does the arguer *want* to reach (directional goal)?
   - Does the arguer construct post-hoc justifications that serve their desired conclusion?
   - Are values being selected because they support the arguer's identity, self-interest, or social goals, rather than because they logically follow from the premise?
3. For each value in the VALUES_LIST (in fixed order), If asked 'Why is this good?', might this be their justification? "Because it is good to "value"" be a coherent justification that **could plausibly arise from motivated reasoning**? Consider:
   - Would a motivated reasoner latch onto this value because it *serves their goal* of justifying their stance?
   - Is this value selected as a post-hoc rationalization rather than a genuine logical derivation?
   - Does this value support the arguer's self-interest, identity, or in-group, while suppressing values that would undermine their desired conclusion?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key-value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# MOTIVATED REASONING EXPERT CONTEXT (Mental Model)
- **Motivated reasoning bias** in reasoning is the tendency to process information, evaluate evidence, and construct arguments in a manner that serves one's pre-existing emotional, identity-based, or self-interested goals, rather than in an objective, truth-seeking manner. People unconsciously reach conclusions they *want* to be true, then work backward to construct seemingly logical justifications.
- **Your expertise includes:** Directional goals (wanting a specific outcome), post-hoc rationalization, biased information search and assimilation, identity-protective cognition, self-serving bias, cognitive dissonance reduction, and the tendency to generate justifications that support pre-existing stances while ignoring or devaluing counterarguments.
- **Your mission:** Identify values that a *motivated* human would select, not values that a *rational* human would select. The arguer may have already decided their stance and is constructing reasons afterwards, may select values that serve their self-interest or identity, may suppress values that would undermine their position, or may genuinely believe their post-hoc rationalizations are objective.

# VALUES_LIST (in fixed order)
 "{VALUES_LIST}"
 
# EXAMPLE OUTPUT (illustrative, not for this input)
{{'Be creative':1, 'Be curious':0, 'Have freedom of thought':0, 'Be choosing own goals':0,..., 'Have the wisdom to accept others':1, 'Be logical':0, 'Have an objective view':0}}

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"
"""

##### Confirmation Bias Prompt

In [22]:
CONFIRMATION_BIAS_PROMPT = """
# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **confirmation bias**.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK
Select for the given argument which of 54 justifications in the VALUE_LIST one could provide for it **considering how confirmation bias might shape or distort that justification**.
**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is influenced by confirmation bias.
**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from confirmatory reasoning**? 

# SELECTION GUIDELINES (CONFIRMATION BIAS-AWARE)
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Read the argument and justification. Select 1 (someone could provide the justification for the argument, even if you may disagree) or 0 (the justification makes no sense for the argument).

# WORKFLOW
1. Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential confirmation biases** that could influence how this argument is formulated and justified.
   - Does the arguer selectively focus on evidence that supports their stance while ignoring counterevidence?
   - Does the premise interpret ambiguous information in a way that favours the arguer's pre-existing conclusion?
   - Is the arguer dismissing or overlooking evidence that contradicts their position?
3. For each value in the VALUES_LIST (in fixed order), If asked 'Why is this good?', might this be their justification? "Because it is good to "value"" be a coherent justification that **could plausibly arise from confirmatory reasoning**? Consider:
   - Would a confirmatory reasoner latch onto this value because it *supports* their pre-existing stance?
   - Does this value align with the arguer's pre-existing worldview or beliefs?
   - Is this value being selected because the arguer is ignoring counter-values that would challenge their stance?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key-value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# CONFIRMATION BIAS EXPERT CONTEXT (Mental Model)
- **Confirmation bias** in reasoning is the tendency to search for, interpret, and recall information in a way that confirms one's pre-existing beliefs or hypotheses, while giving disproportionately less weight to contradictory evidence.
- **Your expertise includes:** Selective attention to supporting evidence, biased interpretation of ambiguous information, belief perseverance, seeking confirming evidence, ignoring disconfirming evidence, assimilation bias, and the tendency to overvalue information that aligns with one's stance.
- **Your mission:** Identify values that a *confirmatory* human would select, not values that a *rational* human would select. The arguer may selectively seek and interpret evidence to support their existing beliefs, may ignore contradictory information, may overvalue information that aligns with their stance, or may genuinely believe their biased interpretation is objective.

# VALUES_LIST (in fixed order)
 "{VALUES_LIST}"
 
# EXAMPLE OUTPUT (illustrative, not for this input)
{{'Be creative':1, 'Be curious':0, 'Have freedom of thought':0, 'Be choosing own goals':0,..., 'Have the wisdom to accept others':1, 'Be logical':0, 'Have an objective view':0}}

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"
"""

##### Hasty Generalisation Prompt

In [23]:
HASTY_GENERALIZATION_PROMPT = """
# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **hasty generalization bias**.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK
Select for the given argument which of 54 justifications in the VALUE_LIST one could provide for it **considering how hasty generalization might shape or distort that justification**.
**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is based on overgeneralization.
**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from overgeneralized reasoning**? 

# SELECTION GUIDELINES (HASTY GENERALIZATION-AWARE)
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Read the argument and justification. Select 1 (someone could provide the justification for the argument, even if you may disagree) or 0 (the justification makes no sense for the argument).

# WORKFLOW
1. Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential hasty generalizations** that could influence how this argument is formulated and justified.
   - Is the premise making a broad claim based on insufficient evidence?
   - Does the argument rely on anecdotes, personal experiences, or vivid examples?
   - Does the premise use absolutist language (e.g., "always," "never," "everyone," "no one")?
3. For each value in the VALUES_LIST (in fixed order), If asked 'Why is this good?', might this be their justification? "Because it is good to "value"" be a coherent justification that **could plausibly arise from overgeneralized reasoning**? Consider:
   - Would an overgeneralizing reasoner latch onto this value because the problem seems *widespread* and *urgent*?
   - Does this value represent a *prevention-focused* response to an overgeneralized threat?
   - Is this value being selected because the arguer has extrapolated from limited examples to universal claims?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key-value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# HASTY GENERALIZATION EXPERT CONTEXT (Mental Model)
- **Hasty generalization bias** in reasoning is the tendency to draw broad conclusions based on insufficient, unrepresentative, or atypical evidence.
- **Your expertise includes:** Overgeneralization, anecdotal reasoning, stereotyping, insufficient sample bias, absolutist thinking, confirmation through limited exposure, and the tendency to extrapolate from personal experience to universal claims.
- **Your mission:** Identify values that a *hastily generalizing* human would select, not values that a *rational* human would select. The arguer may draw sweeping conclusions from limited evidence, may rely on vivid personal stories, may use absolutist language, or may genuinely believe that their limited experience represents the entire population.

# VALUES_LIST (in fixed order)
 "{VALUES_LIST}"
 
# EXAMPLE OUTPUT (illustrative, not for this input)
{{'Be creative':1, 'Be curious':0, 'Have freedom of thought':0, 'Be choosing own goals':0,..., 'Have the wisdom to accept others':1, 'Be logical':0, 'Have an objective view':0}}

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"
"""

##### Causal Oversimplification Prompt -> 1,333 total tokens


In [24]:
CAUSAL_OVERSIMPLIFICATION_PROMPT = """
# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **causal oversimplification bias**.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK
Select for the given argument which of 54 justifications in the VALUE_LIST one could provide for it **considering how causal oversimplification might shape or distort that justification**.
**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is oversimplified.
**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from oversimplified causal reasoning**? 

# SELECTION GUIDELINES (CAUSAL OVERSIMPLIFICATION-AWARE)
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Read the argument and justification. Select 1 (someone could provide the justification for the argument, even if you may disagree) or 0 (the justification makes no sense for the argument).

# WORKFLOW
1. Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential causal oversimplifications** that could influence how this argument is formulated and justified.
   - What complex phenomenon is being reduced to a single cause?
   - What other contributing factors are being ignored?
   - Is the premise treating correlation as causation?
3. For each value in the VALUES_LIST (in fixed order), If asked 'Why is this good?', might this be their justification? "Because it is good to "value"" be a coherent justification that **could plausibly arise from oversimplified causal reasoning**? Consider:
   - Would an oversimplifying reasoner latch onto this value as a *solution* to the problem?
   - Does this value represent a *simplistic fix* rather than a *nuanced approach*?
   - Is this value being selected because the arguer believes the problem is simpler than it actually is?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key-value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# CAUSAL OVERSIMPLIFICATION EXPERT CONTEXT (Mental Model)
- **Causal oversimplification bias** in reasoning is the tendency to attribute complex outcomes to a single, simple cause while ignoring other contributing factors.
- **Your expertise includes:** Single-cause attribution, silver bullet thinking, linear causality, correlation vs. causation, omission of moderators/mediators, reductionist reasoning, and the tendency to oversimplify complex social, technological, or psychological phenomena.
- **Your mission:** Identify values that a *causally oversimplifying* human would select, not values that a *rational* human would select. The arguer may reduce complex problems to simple causes, may ignore systemic factors, may treat correlation as causation, or may genuinely believe that a single intervention will solve everything.

# VALUES_LIST (in fixed order)
 "{VALUES_LIST}"
 
# EXAMPLE OUTPUT (illustrative, not for this input)
{{'Be creative':1, 'Be curious':0, 'Have freedom of thought':0, 'Be choosing own goals':0,..., 'Have the wisdom to accept others':1, 'Be logical':0, 'Have an objective view':0}}

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"
"""

### Experiments with each type of bias

In [25]:
l_prompts = [CAUSAL_OVERSIMPLIFICATION_PROMPT,HASTY_GENERALIZATION_PROMPT,CONFIRMATION_BIAS_PROMPT,FRAMING_EFFECT_PROMPT,MOTIVATED_REASONING_PROMPT]
outputnames = ['causal_oversimplification','hasty_generalization','confirmation_bias','framing_effect','motivated_reasoning']

for bias_prompt, out in zip(l_prompts,outputnames):
    output = f'results/{out}_gpt_oss_503.csv'
    try:
        results = run_experiment_biases(client, model, arguments_503, bias_prompt, output)
    except:
        pass


Resuming... 503 arguments already processed (out of 503).
Resuming... 503 arguments already processed (out of 503).
Resuming... 503 arguments already processed (out of 503).
Resuming... 503 arguments already processed (out of 503).
Resuming... 503 arguments already processed (out of 503).


## Heuristics Toolbox

In [ ]:
HEURISTICS_TOOLBOX = """
In addition to standard reasoning, you must apply all four cognitive heuristics described below. Consider each of them as part of your reasoning process.

1. Representativeness (Problem: Not enough meaning):
Humans tend to interpret situations by matching them to familiar patterns or prototypes when information is incomplete.
- Infer values by comparing the argument to similar known cases.
- Related biases: stereotyping, clustering illusion, illusory correlation, base rate fallacy.

2. Availability (Problem: Too much information):
Humans prioritize information that is more salient, recent, or easy to recall.
- Give more weight to values that are explicit, emotionally striking, or commonly associated with the scenario.
- Related biases: availability heuristic, attentional bias, confirmation bias, framing effect.

3. Anchoring and Adjustment (Problem: Need to act fast):
Humans rely heavily on initial information (anchors) and make only limited adjustments.
- Use the conclusion and premise as the main anchor for interpretation.
- Avoid overthinking beyond what is directly stated.
- Related biases: anchoring bias, status quo bias, conservatism bias.

4. Affect (Problem: What should we remember):
Humans rely on emotional reactions to guide judgment and memory.
- Focus on values related to harm, safety, fairness, well-being, vulnerability, or injustice.
- Related biases: affect heuristic, negativity bias, empathy gap, moral judgment biases.
"""

BASELINE_VALUE_PREDICTION_PROMPT = """
# ROLE & GOAL
You are an expert annotator. Your task is to determine which of 54 human values could reasonably serve as a justification for a given argument.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK:
Select for the given argument which of 54 justifications one could provide for it. 

# SELECTION GUIDELINES:
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Read the argument and justification. Select 1 (someone could provide the justification for the argument, even if you may disagree) or 0 (the justification makes no sense for the argument).

# WORKFLOW
1. Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. For each value in the VALUES_LIST (in fixed order), If asked 'Why is this good?', might this be their justification? "Because it is good to "value"".
3. Assign 1 or 0 accordingly.


# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key-value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# VALUES_LIST (in fixed order)
 "{VALUES_LIST}"
 
# EXAMPLE OUTPUT (illustrative, not for this input)
{{'Be creative':1, 'Be curious':0, 'Have freedom of thought':0, 'Be choosing own goals':0,..., 'Have the wisdom to accept others':1, 'Be logical':0, 'Have an objective view':0}}

 
# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"

"""

In [ ]:
llms = ['gpt-oss-120b']
output_csv = ['results/heuristics_gpt_oss_503.csv']
client = crear_cliente()
for model, output in zip(llms, output_csv):
    results = run_experiment_biases(client, model, arguments_503,HEURISTICS_TOOLBOX+"\n"+BASELINE_VALUE_PREDICTION_PROMPT, output)